# PromptBench

## An Empirical Study of Prompt Strategies for LLM Sentiment Classification

**Author:** Mohammed Mehedi  ·  **Dataset:** IMDb (Hugging Face)  ·  **Provider:** Google Gemini

---

This notebook is the research interface for PromptBench. It does not implement the
experiment — every function it calls lives in `src/` and is covered by the test suite.
That separation is deliberate: logic that lives in a notebook cannot be unit-tested,
reused by the CLI, or trusted by a reader who did not run it.

Read it top to bottom. Cells that spend money are switched off by default and clearly marked.

### Setup

One import block, drawn entirely from the library. If this cell fails, the environment
is not set up — see the README's *Run it locally* section.

In [ ]:
%matplotlib inline

from __future__ import annotations

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# --- Project modules: every piece of logic in this notebook comes from here ---
from src.config import AppConfig, has_api_key
from src.dataset import (
    create_few_shot_examples,
    create_fixed_evaluation_set,
    load_imdb_dataset,
    prepare_dataset,
)
from src.evaluation.errors import (
    error_analysis_summary,
    extract_errors,
    find_shared_errors,
    summarize_errors_by_strategy,
    unique_errors,
)
from src.evaluation.metrics import (
    build_metrics_table,
    confusion_matrix_frame,
    evaluate_strategies,
    rank_strategies,
)
from src.experiments.runner import BenchmarkRunner, build_experiment_summary
from src.experiments.storage import ExperimentStorage, StorageError
from src.llm.gemini import GeminiProvider
from src.prompts import PromptContext, describe_strategies, get_strategy
from src.utils.parsing import parse_sentiment_response
from src.visualization.charts import (
    format_strategy_name,
    plot_accuracy_comparison,
    plot_confusion_matrices,
    plot_error_rate_comparison,
    plot_latency_comparison,
    plot_metric_comparison,
)

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_colwidth", 90)

CONFIG = AppConfig.from_env()
STORAGE = ExperimentStorage(CONFIG.experiments_dir)

print(f"{CONFIG.app_name} ready")
print(f"  model        {CONFIG.model}")
print(f"  temperature  {CONFIG.temperature}")
print(f"  seed         {CONFIG.random_seed}")
print(f"  API key      {'configured' if has_api_key() else 'NOT configured — read-only mode'}")

---

## 1. Executive summary

**The question.** Prompt engineering is discussed anecdotally — "add a role", "give it
examples". This project treats it as a measurable engineering variable and asks how much
the prompt alone moves a benchmark when everything else is pinned down.

**The design.** One model, one dataset, one fixed set of reviews, one parser, one set of
generation settings. Six prompt strategies. The only thing that varies between runs is
the prompt.

**What is reported below** comes from a stored experiment — real predictions, saved to
disk, re-derivable from `predictions.csv`. Where no experiment has been run, the notebook
says *Not yet evaluated* rather than showing a placeholder number.

In [ ]:
def load_latest_experiment() -> dict | None:
    """Load the most recent stored experiment, or None if there are none yet.

    The notebook reads results back from disk rather than holding them in memory,
    so what it displays is exactly what the repository contains.
    """
    experiments = STORAGE.list_experiments()
    if not experiments:
        return None
    experiment_id = experiments[-1]
    try:
        return {
            "experiment_id": experiment_id,
            "config": STORAGE.load_config(experiment_id),
            "predictions": STORAGE.load_predictions(experiment_id),
            "metrics": STORAGE.load_metrics(experiment_id),
            "summary": STORAGE.load_summary(experiment_id),
        }
    except StorageError as error:
        print(f"Experiment {experiment_id} is incomplete: {error}")
        return None


EXPERIMENT = load_latest_experiment()
RESULTS_AVAILABLE = EXPERIMENT is not None

if RESULTS_AVAILABLE:
    totals = EXPERIMENT["summary"]["totals"]
    best = EXPERIMENT["summary"].get("best_strategy")
    print(f"Loaded experiment {EXPERIMENT['experiment_id']}")
    print(f"  {totals['predictions']:,} predictions · {totals['correct']:,} correct "
          f"· {totals['unknown']:,} unparseable · {totals['api_failures']:,} API failures")
    print(f"  Best strategy by F1: {best['strategy'] if best else 'tied — no single winner'}")
else:
    print("Not yet evaluated — no experiment has been run.")
    print("Run one with:  python -m promptbench benchmark --samples 100")

---

## 2. Research question

> **How much can prompt strategy affect the performance of an LLM on sentiment
> classification when the model, dataset, test samples, and evaluation methodology
> remain controlled?**

The controls are the point. Without them, a difference between two prompts could come
from different samples, a different model version, a different temperature, or a more
forgiving parser. Isolating the prompt is what turns an anecdote into a measurement.

A secondary question follows from the first: **do prompt techniques compose?** If a role
helps and demonstrations help, does using both help more? That is what the *combined*
strategy exists to test.

---

## 3. Hypotheses

Each strategy carries a hypothesis written **before any benchmark was run** and stored
in the strategy class itself — so results are read against a prediction rather than
explained after the fact. The table below is generated from those class attributes;
it cannot drift from the code that produced the prompts.

In [ ]:
hypotheses = pd.DataFrame(describe_strategies())[["name", "description", "hypothesis"]]
hypotheses["name"] = hypotheses["name"].map(format_strategy_name)
hypotheses.columns = ["Strategy", "What it does", "Hypothesis (stated in advance)"]
hypotheses.style.hide(axis="index").set_properties(**{"text-align": "left", "font-size": "0.9em"})

**A prediction about the whole experiment**, worth stating plainly so it can be wrong:
the largest single effect will come from *format reliability* rather than *judgement*.
A model that already classifies sentiment well can still lose accuracy by answering in
prose that no parser can read — and that is a failure a prompt can fix outright.

---

## 4. Experimental methodology

A controlled experiment with exactly one independent variable.

| Held constant | Varied |
| --- | --- |
| Dataset and split | **Prompt strategy** |
| The fixed evaluation subset and its ground-truth labels | |
| Model identifier | |
| Temperature and generation settings | |
| Response parser | |
| Metric computation | |
| Random seed | |

**Procedure**

1. Load IMDb; normalise labels to `positive` / `negative`.
2. Draw **one fixed, seeded** evaluation subset. Every strategy is scored on exactly these reviews.
3. Draw **fixed demonstrations** from the *train* split, excluding every evaluation sample.
4. For each strategy: render a prompt per review, call the model with identical settings.
5. Parse each raw response with **one shared parser** into `positive` / `negative` / `unknown`.
6. Score accuracy, precision, recall, F1 and a confusion matrix per strategy.
7. Persist predictions, metrics, errors and metadata under an immutable experiment id.

**Integrity rules that constrain everything below**

- No fabricated numbers. Every metric traces back to a stored raw response.
- Failed API calls and unparseable answers are counted and reported, never dropped.
- Observed results and their interpretation are stated separately.
- No hidden chain-of-thought is requested or stored — only the final label and the raw text.
- One parser for all strategies. A lenient parser would hand its strategy free accuracy.

---

## 5. Dataset

IMDb movie reviews: 25,000 train and 25,000 test, balanced, binary sentiment. It is a
useful benchmark here precisely because it is *not* easy — reviews are long, frequently
mixed ("the acting was superb, the script was not"), and often sarcastic. That is where
prompt design has room to matter.

Loading is cached by Hugging Face, so the download happens once per machine.

In [ ]:
test_pool = prepare_dataset(load_imdb_dataset("test"), split="test")

lengths = test_pool["review"].str.len()
print(f"Prepared test split: {len(test_pool):,} reviews")
print(f"  class balance   {test_pool['label'].value_counts().to_dict()}")
print(f"  review length   mean {lengths.mean():.0f} chars · median {lengths.median():.0f} "
      f"· max {lengths.max():,}")

preview = test_pool.head(3).copy()
preview["review"] = preview["review"].str.slice(0, 110) + "…"
preview

**Why the preparation step exists.** Raw IMDb text carries literal `<br />` markup, which
is noise the model would pay tokens to read. `prepare_dataset` strips it, collapses
whitespace, drops empty and duplicate reviews, and assigns a stable `sample_id` derived
from the *original row position* — so any error row can be traced back to the exact source
record even after filtering.

---

## 6. Controlled variables

Everything below is fixed for the whole run and recorded in `config.json`. Recording them
is what makes a result interpretable six months later, when the hosted model has changed.

In [ ]:
controlled = pd.DataFrame(
    [
        ("Model", CONFIG.model, "Identical for every strategy; hosted models change over time"),
        ("Temperature", CONFIG.temperature, "0.0 is the most repeatable decoding the provider offers"),
        ("Random seed", CONFIG.random_seed, "Same seed + size always selects the same reviews"),
        ("Evaluation samples", CONFIG.benchmark_sample_size, "Bounds how much a difference can be trusted"),
        ("Label vocabulary", " / ".join(CONFIG.labels), "Ground truth uses the words the model is asked to emit"),
        ("Parser", "one shared parser", "No strategy benefits from lenient parsing of its own format"),
        ("Strategies", len(CONFIG.prompt_strategies), "The single independent variable"),
    ],
    columns=["Variable", "Value", "Why it is controlled"],
)
controlled.style.hide(axis="index").set_properties(**{"text-align": "left"})

One further control worth naming: **model-side thinking is disabled**
(`thinking_budget=0`). Left on, the model spends a variable, invisible number of output
tokens reasoning before it answers — which would vary per prompt and confound a
comparison that is supposed to be about the prompt text alone. This is a documented
constraint on model choice, not an incidental setting.

---

## 7. Prompt strategies

Six strategies, each adding **exactly one mechanism** to the baseline. They differ
structurally, not by wording — a test asserts that demonstrations appear only in the
few-shot family, a persona only in the role family, and so on.

In [ ]:
mechanisms = pd.DataFrame(
    [
        ("Zero-shot",   "—",                    "Control condition: task, labels, review"),
        ("Few-shot",    "In-context learning",  "Fixed, class-balanced solved examples"),
        ("Role-based",  "Persona + guideline",  "Expert annotator in the system slot, with its rules"),
        ("Structured",  "Output contract",      "A single JSON object, no fences, no prose"),
        ("Reasoning",   "Decision procedure",   "Four-point checklist, silent; label only"),
        ("Combined",    "Composition",          "Role + demonstrations + procedure + JSON"),
    ],
    columns=["Strategy", "Added mechanism", "How it works"],
)
mechanisms.style.hide(axis="index").set_properties(**{"text-align": "left"})

**On chain-of-thought.** The reasoning strategy asks the model to weigh the evidence
*silently* and return only the verdict. No reasoning is requested, returned or stored —
`predictions.csv` contains the label and the raw response, nothing hidden.

**On the structured strategy.** Its contract is expressed in prompt text only.
Provider-side constrained decoding would change *generation settings* and stop this from
being a prompt-only comparison, so it is deliberately not used.

---

## 8. Prompt examples

The clearest way to see that the strategies genuinely differ is to render them all against
the same review and compare. The demonstrations used here are the same fixed ones the
benchmark uses.

In [ ]:
evaluation_set = create_fixed_evaluation_set(
    test_pool, sample_size=CONFIG.benchmark_sample_size, seed=CONFIG.random_seed
)
train_pool = prepare_dataset(load_imdb_dataset("train"), split="train")
few_shot_examples = tuple(
    create_few_shot_examples(
        train_pool,
        examples_per_class=2,
        seed=CONFIG.random_seed,
        exclude_sample_ids=evaluation_set["sample_id"],
    )
)
context = PromptContext(few_shot_examples=few_shot_examples)

leakage = set(evaluation_set["review"]) & {example.review for example in few_shot_examples}
print(f"Fixed evaluation set : {len(evaluation_set)} reviews "
      f"({evaluation_set['label'].value_counts().to_dict()})")
print(f"Fixed demonstrations : {len(few_shot_examples)} "
      f"({[example.label for example in few_shot_examples]})")
print(f"Leakage check        : {len(leakage)} demonstrations appear in the evaluation set")

In [ ]:
SAMPLE_REVIEW = "Slow in places and the middle act drags, but the ending completely won me over."

sizes = []
for strategy_name in [str(member) for member in CONFIG.prompt_strategies]:
    prompt = get_strategy(strategy_name).build_prompt(SAMPLE_REVIEW, context)
    sizes.append(
        {
            "Strategy": format_strategy_name(strategy_name),
            "Characters": prompt.char_count,
            "System slot": "yes" if prompt.system_instruction else "—",
            "Demonstrations": "yes" if "<example>" in prompt.user_text else "—",
            "JSON contract": "yes" if '"sentiment"' in prompt.user_text else "—",
            "Decision procedure": "yes" if "silently" in prompt.user_text else "—",
        }
    )
pd.DataFrame(sizes).style.hide(axis="index").set_properties(**{"text-align": "left"})

Each row differs from the baseline by one column. That is the experiment's structure made
visible: any measured difference is attributable to the mechanism that column represents.

Below, two prompts in full — the shortest and the longest.

In [ ]:
for strategy_name in ("zero_shot", "combined"):
    prompt = get_strategy(strategy_name).build_prompt(SAMPLE_REVIEW, context)
    print("=" * 84)
    print(f"  {format_strategy_name(strategy_name)}  ·  {prompt.char_count:,} characters")
    print("=" * 84)
    if prompt.system_instruction:
        print("[system instruction]")
        print(prompt.system_instruction)
        print()
    print("[user turn]")
    print(prompt.user_text)
    print()

---

## 9. Single prediction demonstration

Before spending quota on a full benchmark, verify the whole path on one review: prompt in,
raw text out, parsed label, latency, tokens. This is also the cell that shows what the
parser is actually doing.

> **This cell calls the API.** It is off by default. Set `RUN_LIVE_DEMO = True` to enable it
> — it costs one call per strategy.

In [ ]:
RUN_LIVE_DEMO = False   # ← set True to make real API calls (one per strategy)

DEMO_SAMPLE = evaluation_set.iloc[0]
print(f"Sample  : {DEMO_SAMPLE['sample_id']}")
print(f"True    : {DEMO_SAMPLE['label']}")
print(f"Review  : {DEMO_SAMPLE['review'][:220]}…")
print()

if not RUN_LIVE_DEMO:
    print("Live demo disabled — set RUN_LIVE_DEMO = True to call the API.")
elif not has_api_key():
    print("No GEMINI_API_KEY configured; cannot call the model.")
else:
    provider = GeminiProvider(CONFIG)
    rows = []
    for strategy_name in [str(member) for member in CONFIG.prompt_strategies]:
        prompt = get_strategy(strategy_name).build_prompt(DEMO_SAMPLE["review"], context)
        response = provider.generate(prompt)
        parsed = parse_sentiment_response(response.text)
        rows.append(
            {
                "Strategy": format_strategy_name(strategy_name),
                "Raw response": (response.text or "(empty)")[:46],
                "Parsed": parsed,
                "Correct": "yes" if parsed == DEMO_SAMPLE["label"] else "no",
                "Latency": f"{response.latency_seconds:.2f}s",
                "Tokens": response.usage.total_tokens,
            }
        )
    display(pd.DataFrame(rows).style.hide(axis="index"))

The **Raw response** column next to **Parsed** is the honest view: it shows what the model
actually said and what the shared parser made of it. A response the parser cannot read
becomes `unknown` — counted as a reliability failure, never guessed at.

---

## 10. Benchmark configuration

What the run will cost, stated before it starts.

In [ ]:
strategy_count = len(CONFIG.prompt_strategies)
sample_count = len(evaluation_set)
planned_calls = strategy_count * sample_count

print(f"Strategies      {strategy_count}")
print(f"Samples each    {sample_count}   (identical for every strategy)")
print(f"Total API calls {planned_calls:,}")
print(f"Model           {CONFIG.model} @ temperature {CONFIG.temperature}")
print(f"Seed            {CONFIG.random_seed}")
print()
print("Sample ids are checksummed into config.json, so two runs can be proven to have")
print("been scored on identical reviews.")

---

## 11. Benchmark execution

The runner sends every strategy through the same samples, in the same order, with the same
settings, recording the outcome of every call — including the ones that fail. A failed call
still produces a row with `predicted_label = unknown`, because dropping it would quietly
shrink the denominator of every metric below.

> **This cell spends real quota.** It is off by default. Either set `RUN_BENCHMARK = True`,
> or run it from the terminal — which is the recommended path, since it survives a closed
> notebook:
>
> ```bash
> python -m promptbench benchmark --samples 100
> ```
>
> Either way the notebook picks the results up from `results/experiments/`.

In [ ]:
RUN_BENCHMARK = False   # ← set True to run the full benchmark from this notebook

if RUN_BENCHMARK and has_api_key():
    runner = BenchmarkRunner(GeminiProvider(CONFIG), CONFIG, storage=STORAGE)
    result = runner.run(evaluation_set, list(CONFIG.prompt_strategies), few_shot_examples)
    evaluation = runner.evaluate(result)

    EXPERIMENT = {
        "experiment_id": result.experiment_id,
        "config": result.config,
        "predictions": result.predictions,
        "metrics": evaluation.metrics,
        "summary": evaluation.summary,
    }
    RESULTS_AVAILABLE = True
    print(f"Finished {result.experiment_id} in {result.runtime_seconds:.0f}s")
elif RUN_BENCHMARK:
    print("No GEMINI_API_KEY configured; cannot run the benchmark.")
else:
    print("Benchmark not run from the notebook.")
    if RESULTS_AVAILABLE:
        print(f"Using stored experiment {EXPERIMENT['experiment_id']} "
              f"from {CONFIG.experiments_dir}")
    else:
        print("No stored experiment found — the sections below will report "
              "'Not yet evaluated'.")

In [ ]:
def require_results(section: str) -> bool:
    """Guard every results section so the notebook renders honestly when empty.

    An empty section that says so is informative; one filled with placeholder
    numbers is not.
    """
    if RESULTS_AVAILABLE:
        return True
    print(f"{section}: Not yet evaluated.")
    print("Run:  python -m promptbench benchmark --samples 100")
    return False


if RESULTS_AVAILABLE:
    METRICS = EXPERIMENT["metrics"]
    PREDICTIONS = EXPERIMENT["predictions"]
    SUMMARY = EXPERIMENT["summary"]
    ERRORS = extract_errors(PREDICTIONS)
    print(f"Working from experiment {EXPERIMENT['experiment_id']}: "
          f"{len(PREDICTIONS):,} predictions across "
          f"{PREDICTIONS['strategy'].nunique()} strategies")
else:
    METRICS = PREDICTIONS = SUMMARY = ERRORS = None

---

## 12. Results table

One row per strategy. `accuracy` counts an unparseable answer as **incorrect** — a response
nobody can read is not a correct classification. `unknown_rate` is reported beside it
because format compliance is one of the things this benchmark measures.

In [ ]:
if require_results("Results table"):
    columns = ["strategy", "correct", "incorrect", "unknown", "accuracy",
               "precision_macro", "recall_macro", "f1_macro",
               "error_rate", "unknown_rate", "avg_latency_seconds"]
    table = METRICS[[column for column in columns if column in METRICS.columns]].copy()
    table["strategy"] = table["strategy"].map(format_strategy_name)
    display(
        table.style.hide(axis="index")
        .format({"accuracy": "{:.1%}", "precision_macro": "{:.3f}",
                 "recall_macro": "{:.3f}", "f1_macro": "{:.3f}",
                 "error_rate": "{:.1%}", "unknown_rate": "{:.1%}",
                 "avg_latency_seconds": "{:.2f}s"})
        .background_gradient(subset=["f1_macro"], cmap="Blues")
    )

**How to read precision and recall here.** An `unknown` is treated as *no prediction made*:
it costs recall on the true class but does not pollute precision on either. That is the
standard treatment for an abstention, and it has a consequence worth holding onto —
**abstaining scores better than being wrong**. Read F1 next to `unknown_rate`, never alone.

---

## 13. Accuracy comparison

The headline measure: of everything the strategy was asked, how much did it get right.

In [ ]:
if require_results("Accuracy comparison"):
    plt.close("all")
    display(plot_accuracy_comparison(METRICS))

---

## 14. Precision, recall and F1

Accuracy alone hides *how* a strategy is wrong. A strategy that never predicts `negative`
can still post a respectable accuracy on a balanced set; per-class precision and recall
expose that immediately.

In [ ]:
if require_results("Precision / recall / F1"):
    plt.close("all")
    display(plot_metric_comparison(METRICS))

In [ ]:
if require_results("Per-class detail"):
    per_class = METRICS[["strategy", "precision_negative", "recall_negative", "f1_negative",
                         "precision_positive", "recall_positive", "f1_positive"]].copy()
    per_class["strategy"] = per_class["strategy"].map(format_strategy_name)
    display(
        per_class.style.hide(axis="index")
        .format({column: "{:.3f}" for column in per_class.columns if column != "strategy"})
    )

---

## 15. Confusion matrices

Where each strategy's errors actually land. The `unknown` column is always present, even
when empty, so the panels have identical shape and can be compared directly; the colour
scale is shared for the same reason.

In [ ]:
if require_results("Confusion matrices"):
    plt.close("all")
    display(plot_confusion_matrices(PREDICTIONS))

---

## 16. Latency

Cost, not quality. A strategy that wins by a point but takes twice as long per call is a
different engineering trade-off, and a reader deciding what to deploy needs to see it.
Longer prompts also cost more input tokens on every single call.

In [ ]:
if require_results("Latency"):
    plt.close("all")
    display(plot_latency_comparison(METRICS))

In [ ]:
if require_results("Cost view"):
    cost = METRICS[["strategy", "accuracy", "f1_macro", "avg_latency_seconds", "runtime_seconds"]].copy()
    if "total_tokens" in PREDICTIONS.columns:
        tokens = PREDICTIONS.groupby("strategy")["total_tokens"].sum()
        cost["total_tokens"] = cost["strategy"].map(tokens)
        # Cast to float first: Arrow-backed integer columns cannot divide by a
        # masked value, and a strategy with zero correct answers must not
        # produce a division error or a misleading infinity.
        correct = METRICS["correct"].astype("float64")
        cost["tokens_per_correct"] = (
            cost["total_tokens"].astype("float64") / correct.where(correct > 0)
        ).round(0)
    cost["strategy"] = cost["strategy"].map(format_strategy_name)
    display(cost.style.hide(axis="index").format({
        "accuracy": "{:.1%}", "f1_macro": "{:.3f}",
        "avg_latency_seconds": "{:.2f}s", "runtime_seconds": "{:.1f}s",
    }, na_rep="—"))

---

## 17. Error analysis

Every incorrect prediction is stored with the review, both labels and the raw response —
which is what makes a claim about *why* a strategy failed checkable rather than asserted.

Errors are split by cause, because the two call for completely different fixes:

- **Misclassification** — the model produced a readable label, and it was the wrong one.
  A judgement failure.
- **Unparseable** — no label could be extracted at all. A format failure.

In [ ]:
if require_results("Error analysis"):
    overview = error_analysis_summary(ERRORS)
    print(f"Total errors     {overview['total_errors']}")
    print(f"  misclassified  {overview['misclassifications']}")
    print(f"  unparseable    {overview['unparseable']}")
    print(f"Samples affected {overview['samples_with_errors']}")
    print()
    breakdown = summarize_errors_by_strategy(ERRORS).copy()
    breakdown["strategy"] = breakdown["strategy"].map(format_strategy_name)
    display(breakdown.style.hide(axis="index"))

In [ ]:
if require_results("Error rate by cause"):
    plt.close("all")
    display(plot_error_rate_comparison(METRICS))

### Samples every strategy got wrong

These rows are the most informative in the whole experiment. A review that **no prompt
design rescues** is evidence about the *data* — genuine ambiguity, or a questionable gold
label — rather than about any strategy. Reading a handful of them is usually worth more
than another point of accuracy.

In [ ]:
if require_results("Shared errors"):
    shared = find_shared_errors(ERRORS, strategy_count=PREDICTIONS["strategy"].nunique())
    if shared.empty:
        print("No sample defeated every strategy.")
    else:
        print(f"{len(shared)} sample(s) were misjudged by every strategy:\n")
        for _, row in shared.head(3).iterrows():
            print(f"  {row['sample_id']}  (true label: {row['actual_label']})")
            print(f"  {row['review'][:400]}…")
            print()
        display(shared)

In [ ]:
if require_results("Unique errors"):
    print("Samples that only ONE strategy got wrong — what each design uniquely breaks:\n")
    for strategy_name in PREDICTIONS["strategy"].drop_duplicates():
        only = unique_errors(ERRORS, strategy_name)
        print(f"  {format_strategy_name(strategy_name):<14} {len(only)}")

In [ ]:
if require_results("Error browser"):
    browser = ERRORS.copy()
    browser["review"] = browser["review"].str.slice(0, 100) + "…"
    browser["raw_response"] = browser["raw_response"].astype(str).str.slice(0, 40)
    browser["strategy"] = browser["strategy"].map(format_strategy_name)
    display(browser[["sample_id", "strategy", "actual_label", "predicted_label",
                     "error_type", "raw_response", "review"]].head(20))

---

## 18. Strategy comparison

Ranked by F1, with the abstention rate alongside so the ranking cannot be read naively.

In [ ]:
if require_results("Strategy comparison"):
    ranked = rank_strategies(METRICS, metric="f1_macro")[
        ["rank", "strategy", "f1_macro", "accuracy", "unknown_rate", "avg_latency_seconds"]
    ].copy()
    ranked["strategy"] = ranked["strategy"].map(format_strategy_name)
    display(ranked.style.hide(axis="index").format({
        "f1_macro": "{:.3f}", "accuracy": "{:.1%}",
        "unknown_rate": "{:.1%}", "avg_latency_seconds": "{:.2f}s",
    }))

In [ ]:
if require_results("Deltas against the baseline"):
    baseline = METRICS[METRICS["strategy"] == "zero_shot"]
    if baseline.empty:
        print("The zero-shot baseline was not part of this run, so deltas are undefined.")
    else:
        base = baseline.iloc[0]
        deltas = METRICS[["strategy"]].copy()
        for column in ("accuracy", "f1_macro", "unknown_rate"):
            deltas[f"Δ {column}"] = METRICS[column] - base[column]
        deltas["strategy"] = deltas["strategy"].map(format_strategy_name)
        display(deltas.style.hide(axis="index").format({
            "Δ accuracy": "{:+.1%}", "Δ f1_macro": "{:+.3f}", "Δ unknown_rate": "{:+.1%}",
        }))
        print("\nEach row is that strategy's effect relative to the control condition —")
        print("the quantity this experiment exists to measure.")

---

## 19. Findings

**Observed result** and **interpretation** are kept separate below. The observations are
generated from the stored metrics and cannot be edited by hand; the interpretation is
written by a human and is explicitly labelled as such.

In [ ]:
if require_results("Findings"):
    best = SUMMARY.get("best_strategy")
    ranked = rank_strategies(METRICS, metric="f1_macro")
    baseline = METRICS[METRICS["strategy"] == "zero_shot"]

    print("OBSERVED")
    print("-" * 72)
    print(f"Samples per strategy      {int(METRICS['total_samples'].iloc[0])}")
    print(f"Strategies compared       {len(METRICS)}")
    if best:
        print(f"Highest F1                {best['strategy']} ({best['f1_macro']:.3f})")
    else:
        print(f"Highest F1                tied — no single winner")
    print(f"F1 range across strategies {METRICS['f1_macro'].min():.3f} – {METRICS['f1_macro'].max():.3f}")
    print(f"Accuracy range             {METRICS['accuracy'].min():.1%} – {METRICS['accuracy'].max():.1%}")
    print(f"Unparseable rate range     {METRICS['unknown_rate'].min():.1%} – {METRICS['unknown_rate'].max():.1%}")
    if not baseline.empty:
        base = baseline.iloc[0]
        spread = METRICS["accuracy"].max() - base["accuracy"]
        print(f"Best improvement over the zero-shot baseline: {spread:+.1%} accuracy")
    print(f"Total API failures         {int(METRICS['api_failures'].fillna(0).sum())}")
    print()
    print("Full ranking by F1:")
    for _, row in ranked.iterrows():
        print(f"  {row['rank']}. {format_strategy_name(row['strategy']):<14} "
              f"F1 {row['f1_macro']:.3f}  ·  accuracy {row['accuracy']:.1%}  ·  "
              f"unparseable {row['unknown_rate']:.1%}")

### Interpretation

*Written after reading the numbers above. Fill this in once a full benchmark has been run —
it should not be pre-written, and each claim below must point at a number in the tables.*

Questions worth answering here, each supported by a figure from this notebook:

1. **Did prompt strategy matter at all?** Compare the F1 range against the baseline. A
   spread of a point or two on 100 samples is inside the noise; a spread of ten is not.
2. **Was the effect judgement or format?** Compare `accuracy` against
   `accuracy_on_resolved`. If they diverge, the strategy changed how *readable* the answers
   were, not how *correct* the reasoning was — a different finding, and often the larger one.
3. **Did the techniques compose?** Compare *combined* against its best single component.
   If it underperforms, the instructions likely diluted each other — which is a real,
   publishable result, not a failure of the experiment.
4. **What did the shared errors look like?** If most surviving errors are genuinely
   ambiguous reviews, the ceiling is the *dataset*, not the prompt.
5. **Was the winner worth its cost?** Cross-reference the latency and token table.

Claims not supported by a number in this notebook do not belong in this section.

---

## 20. Limitations

Stated plainly, because a result whose limits are hidden is not a result.

1. **Single model, single task.** These findings describe one Gemini model on IMDb
   sentiment. They do not transfer automatically to other models, domains or task types.
2. **Sample size.** A modest evaluation subset keeps the run affordable, but differences of
   a few points may be inside the noise. No significance test is computed here, so
   small gaps should be read as *suggestive*, not established. Bootstrap confidence
   intervals are the obvious next step.
3. **Non-determinism.** Even at temperature 0, hosted models are not guaranteed to be
   reproducible. Re-running the same experiment may shift the numbers slightly; this is
   reported rather than hidden.
4. **Binary labels on mixed reviews.** IMDb reviews are long and frequently mixed. A
   two-class gold label cannot represent genuine ambiguity, and some "errors" are
   disagreements with a questionable label rather than model failures — which the shared
   error analysis surfaces directly.
5. **The prompt space is unbounded.** Six strategies are a considered sample of a very
   large space, not its optimum. A better prompt for any of these mechanisms certainly exists.
6. **One provider.** Results may partly reflect how this model was trained to follow
   instructions rather than a general property of the technique.
7. **Demonstrations are fixed, not tuned.** Few-shot performance depends on *which*
   examples are shown. These were drawn once by seed and never optimised — deliberately,
   to avoid tuning one strategy while leaving the others untouched.

---

## 21. Conclusion

This project set out to measure whether prompt strategy is an engineering variable worth
controlling, rather than a matter of taste. The apparatus built to answer that — a fixed
seeded evaluation set, one shared parser, immutable experiment records, and a metric layer
that treats unreadable answers as their own category — is what makes the answer
trustworthy in either direction.

Two things are worth stating regardless of what the numbers say:

- **A null result would still be a result.** If six genuinely different prompts land within
  a point of each other, that is useful evidence against a widely repeated claim.
- **Format reliability is a first-class outcome.** A strategy that improves nothing about
  the model's judgement but eliminates unparseable answers has still improved the system,
  and the metrics here are built to show that separately rather than blurring it into
  accuracy.

Fill in the interpretation in §19 once a benchmark has run, and cite the figures.

---

## 22. Reproducibility

Every run writes an immutable directory. Nothing is overwritten, so a result stays
comparable to the run that produced it.

```
results/experiments/<experiment_id>/
├── config.json        # model, temperature, seed, sample ids + checksum, strategies
├── predictions.csv    # one row per (strategy, sample), with the raw response
├── metrics.csv        # per-strategy scores, re-derivable from predictions.csv
├── summary.json       # headline result and ranking
└── charts/            # the figures above, as PNGs
```

In [ ]:
if RESULTS_AVAILABLE:
    stored = EXPERIMENT["config"]
    dataset = stored.get("dataset", {})
    print(f"Experiment      {stored.get('experiment_id')}")
    print(f"Run at          {stored.get('timestamp')}")
    print(f"Model           {stored.get('provider', {}).get('model')} "
          f"@ temperature {stored.get('provider', {}).get('temperature')}")
    print(f"Dataset         {dataset.get('name')} · {dataset.get('sample_count')} samples")
    print(f"Random seed     {dataset.get('random_seed')}")
    print(f"Sample checksum {dataset.get('sample_id_checksum')}")
    print()
    print("Two runs sharing that checksum were scored on exactly the same reviews —")
    print("which is what makes their numbers comparable.")
else:
    print("Not yet evaluated — no experiment metadata to display.")

### Reproducing this study

```bash
git clone <repository-url> && cd <repository-folder>
python3 -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt
cp .env.example .env          # add your GEMINI_API_KEY

pytest                        # 690+ tests, no API calls
python -m promptbench benchmark --samples 100
```

Then re-run this notebook: it picks up the newest experiment from
`results/experiments/` automatically.

**What is guaranteed.** The same seed and sample size always select the same reviews
(checksummed above), all strategies are scored on identical samples, metrics are
re-derivable from `predictions.csv`, and no historical experiment is ever overwritten.

**What is not.** Hosted model outputs are not guaranteed identical between runs even at
temperature 0, so predictions may shift slightly. That variance is a property of the
system under study, and reporting it is part of the result.

---

*Built with the modules in `src/` — dataset, prompts, llm, evaluation, experiments,
visualization — each unit-tested independently. The same code backs the CLI
(`python -m promptbench`) and the dashboard (`streamlit run app/streamlit_app.py`).*